# Final Summary

This notebook is the final project wrap-up. It summarizes the fraud-detection problem, the data preparation decisions, the final selected feature set, the decision system design, and the planned model-evaluation direction. It does not repeat the full analysis from earlier notebooks.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path("..").resolve()
REPORTS_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
REPORTS_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

NOTEBOOK_TABLES_DIR = REPORTS_TABLES_DIR / "12_final_summary"
NOTEBOOK_FIGURES_DIR = REPORTS_FIGURES_DIR / "12_final_summary"
NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tables dir  : {NOTEBOOK_TABLES_DIR}")
print(f"Figures dir : {NOTEBOOK_FIGURES_DIR}")


Tables dir  : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/12_final_summary
Figures dir : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/figures/12_final_summary


## 1. Project Overview

- Problem: detect whether a credit-card transaction is fraudulent (`Class = 1`) or legitimate (`Class = 0`).
- Business goal: identify fraudulent transactions early enough to support `BLOCK`, `REVIEW`, and `APPROVE` actions.
- Dataset: anonymized PCA-based transaction data with `Time`, `V1` to `V28`, `Amount`, and `Class`.
- Project outcome so far: the data is cleaned, the feature set is finalized, and the decision logic is defined for the next modeling phase.


In [4]:
duplicate_summary = pd.read_csv(REPORTS_TABLES_DIR / "04_data_cleaning" / "duplicate_removal_summary.csv")
target_distribution = pd.read_csv(REPORTS_TABLES_DIR / "05_univariate_analysis" / "target_distribution_after_cleaning.csv")
selected_features = pd.read_csv(REPORTS_TABLES_DIR / "10_feature_selection" / "selected_feature_list.csv")
final_dataset_summary = pd.read_csv(REPORTS_TABLES_DIR / "11_eda_insights" / "final_dataset_summary.csv")
decision_system_feature_roles = pd.read_csv(REPORTS_TABLES_DIR / "11_eda_insights" / "decision_system_feature_roles.csv")
decision_threshold_framework = pd.read_csv(REPORTS_TABLES_DIR / "11_eda_insights" / "decision_threshold_framework.csv")
final_recommendation = pd.read_csv(REPORTS_TABLES_DIR / "11_eda_insights" / "final_recommendation_summary.csv")

duplicate_lookup = duplicate_summary.set_index("metric")["value"].to_dict()
fraud_row = target_distribution.loc[target_distribution["Class"] == 1].iloc[0]

project_summary = pd.DataFrame(
    [
        {
            "summary_area": "dataset_status",
            "value": f"{int(duplicate_lookup['rows_after_cleaning'])} cleaned rows after removing {int(duplicate_lookup['rows_removed_as_exact_duplicates'])} exact duplicates.",
        },
        {
            "summary_area": "fraud_rate_after_cleaning",
            "value": f"{fraud_row['percent']:.4f}% fraud rate after cleaning.",
        },
        {
            "summary_area": "final_feature_count",
            "value": str(int(len(selected_features))),
        },
        {
            "summary_area": "modeling_readiness",
            "value": "Feature selection is complete and the dataset is ready for baseline modeling.",
        },
    ]
)

project_summary.to_csv(NOTEBOOK_TABLES_DIR / "project_summary.csv", index=False)
project_summary


,summary_area,value
0,dataset_status,283726 cleaned rows after removing 1081 exact ...
1,fraud_rate_after_cleaning,0.1667% fraud rate after cleaning.
2,final_feature_count,13
3,modeling_readiness,Feature selection is complete and the dataset ...


## 2. Data Preparation Summary

- Duplicate transactions were removed before modeling to reduce leakage risk and repeated-pattern bias.
- No major missing-value problem remained after validation.
- Fraud stayed extremely rare after cleaning, so the project must treat imbalance as a core modeling constraint.
- Transaction amount was strongly skewed, which justified keeping `log_amount` instead of raw `Amount`.


## 3. Final Feature Set

The final selected dataset contains `13` predictors and is the canonical modeling-ready feature space for the baseline models.


In [5]:
selected_features


,feature,feature_category,selection_decision,selection_reason
0,V14_V12_interaction,INTERACTION,KEEP,Shows strong fraud relevance and remains compe...
1,V14,PCA,KEEP,Shows strong fraud relevance and remains compe...
2,V17_V16_interaction,INTERACTION,KEEP,Shows strong fraud relevance and remains compe...
3,V12,PCA,KEEP,Shows strong fraud relevance and remains compe...
4,V17,PCA,KEEP,Shows strong fraud relevance and remains compe...
5,V10,PCA,KEEP,Shows strong fraud relevance and remains compe...
6,V4,PCA,KEEP,Shows strong fraud relevance and remains compe...
7,V16,PCA,KEEP,Shows strong fraud relevance and remains compe...
8,V3,PCA,KEEP,Shows strong fraud relevance and remains compe...
9,V11,PCA,KEEP,Shows strong fraud relevance and remains compe...


### Final Dataset Summary

- Number of features: `13`
- Selected features: `V14_V12_interaction`, `V14`, `V17_V16_interaction`, `V12`, `V17`, `V10`, `V4`, `V16`, `V3`, `V11`, `V7`, `V18`, `log_amount`
- Why this set: it preserves the strongest fraud signal while removing the most problematic redundant features.


## 3. Handling Class Imbalance

- The dataset is highly imbalanced, with fraud making up about `0.17%` of cleaned transactions.
- Strategy used for the modeling phase:
- `class_weight='balanced'` to make fraud errors count more during training.
- Threshold tuning to support fraud-focused `BLOCK`, `REVIEW`, and `APPROVE` decisions instead of relying on a default cutoff.

Impact on model behavior:

- The model becomes more sensitive to rare fraud cases instead of favoring the majority non-fraud class.
- This usually increases fraud recall, but it can also increase false positives and manual review volume.
- The project therefore prioritizes recall-first behavior and then controls operational cost through threshold design.


## 4. Decision System Summary

This project is not only a classifier. It is intended to support fraud operations through `BLOCK`, `REVIEW`, and `APPROVE` decisions.


In [6]:
decision_system_feature_roles


,decision_role,features,why
0,PRIMARY_BLOCK_SIGNAL,"V14_V12_interaction, V14, V17_V16_interaction,...",These features show the strongest fraud separa...
1,SUPPORTING_REVIEW_SIGNAL,"V4, V16, V3, V11, V7, V18, log_amount",These features add supporting context and help...


In [7]:
decision_threshold_framework


,decision_action,probability_rule,decision_reason
0,BLOCK,> 0.85,Very high estimated fraud risk should trigger ...
1,REVIEW,0.60 to 0.85,Intermediate risk should be reviewed manually ...
2,APPROVE,< 0.60,Low estimated fraud risk should be approved un...


### Operational Interpretation

- `BLOCK`: very high predicted fraud probability plus strong primary fraud signals.
- `REVIEW`: moderate or uncertain probability plus meaningful supporting fraud signals.
- `APPROVE`: weak fraud probability without meaningful high-risk signal activation.

Exact probability thresholds:

- `BLOCK`: `> 0.85`
- `REVIEW`: `0.60 to 0.85`
- `APPROVE`: `< 0.60`

Why high recall matters:

- In fraud detection, missing a fraudulent transaction is usually more costly than sending an extra legitimate transaction for review.
- The system should therefore favor catching as many fraud cases as possible, even if that increases manual review volume.

Why the threshold is not `0.50`:

- The dataset is extremely imbalanced, so a default `0.50` cutoff does not reflect the real fraud-detection objective.
- The decision system needs stricter operational zones for `BLOCK`, `REVIEW`, and `APPROVE`, rather than a single generic classification cutoff.


## 6. Model Evaluation Results

⚠️ Model training and evaluation will be completed in the next phase.

Planned models:

- Logistic Regression
- Random Forest

Metrics to evaluate:

- Precision
- Recall
- F1-score
- ROC-AUC

Final model selection will be based on recall (fraud detection priority).


In [ ]:
# Model evaluation outputs will be added in the next phase.


In [ ]:
# Threshold comparison outputs will be added after model validation.


## 7. Final Recommendation

- Top predictive features: `V14_V12_interaction`, `V14`, `V17_V16_interaction`, `V12`, `V17`, and `V10` should anchor fraud-risk scoring.
- Main data challenges: severe class imbalance, skewed transaction amount before transformation, and redundancy across several engineered features.
- Recommended next modeling step: train Logistic Regression and Random Forest on the selected 13-feature dataset.
- Main evaluation goal: maximize fraud recall without ignoring precision, because missed fraud is costlier than reviewing extra suspicious transactions.
- System goal: convert predicted fraud probability into `BLOCK`, `REVIEW`, and `APPROVE` decisions using validated thresholds.


In [8]:
final_recommendation


,recommendation_area,recommendation
0,top_predictive_features,"Use V14_V12_interaction, V14, V17_V16_interact..."
1,data_challenges,"Address severe class imbalance, amount skew be..."
2,modeling_strategy,"Use the selected features, scale where require..."
3,expected_outcome,Prioritize fraud recall and convert validated ...


In [9]:
final_summary_report = f"""# Final Summary Report

## Project Overview

- Problem: credit card fraud detection.
- Goal: classify transactions and support `BLOCK`, `REVIEW`, and `APPROVE` decisions.
- Current status: data preparation, EDA synthesis, feature engineering review, feature selection, and decision design are complete.

## Final Dataset

- Cleaned rows: {int(duplicate_lookup['rows_after_cleaning'])}
- Fraud rate after cleaning: {fraud_row['percent']:.4f}%
- Selected feature count: {int(len(selected_features))}
- Selected features: {', '.join(selected_features['feature'].tolist())}

## Decision System

- Primary high-risk features support `BLOCK`.
- Supporting fraud features support `REVIEW`.
- Weak signal supports `APPROVE`.
- Exact thresholds: `BLOCK > 0.85`, `REVIEW 0.60 to 0.85`, `APPROVE < 0.60`.
- High recall matters because missed fraud is more costly than additional manual review.
- The threshold is not `0.50` because fraud detection is highly imbalanced and requires operational decision bands rather than a default binary cutoff.

## Model Evaluation Results

⚠️ Model training and evaluation will be completed in the next phase.

Planned models:
- Logistic Regression
- Random Forest

Metrics to evaluate:
- Precision
- Recall
- F1-score
- ROC-AUC

Final model selection will be based on recall (fraud detection priority).

## Final Recommendation

- Use the finalized non-redundant feature set as the shared starting point for both baseline models.
- Train Logistic Regression and Random Forest in the next phase and compare them under the same fraud-focused metrics.
- Prioritize fraud recall and decision-threshold calibration.

## Saved Tables

- `reports/tables/12_final_summary/project_summary.csv`
- `reports/tables/12_final_summary/final_summary_report.md`
"""

report_path = NOTEBOOK_TABLES_DIR / "final_summary_report.md"
display(Markdown(final_summary_report))
report_path.write_text(final_summary_report, encoding="utf-8")
print(f"Final summary report saved to: {report_path}")


# Final Summary Report

## Project Overview

- Problem: credit card fraud detection.
- Goal: classify transactions and support `BLOCK`, `REVIEW`, and `APPROVE` decisions.
- Current status: data preparation, EDA synthesis, feature engineering review, feature selection, and decision design are complete.

## Final Dataset

- Cleaned rows: 283726
- Fraud rate after cleaning: 0.1667%
- Selected feature count: 13
- Selected features: V14_V12_interaction, V14, V17_V16_interaction, V12, V17, V10, V4, V16, V3, V11, V7, V18, log_amount

## Decision System

- Primary high-risk features support `BLOCK`.
- Supporting fraud features support `REVIEW`.
- Weak signal supports `APPROVE`.
- Exact thresholds: `BLOCK > 0.85`, `REVIEW 0.60 to 0.85`, `APPROVE < 0.60`.
- High recall matters because missed fraud is more costly than additional manual review.
- The threshold is not `0.50` because fraud detection is highly imbalanced and requires operational decision bands rather than a default binary cutoff.

## Model Evaluation Results

⚠️ Model training and evaluation will be completed in the next phase.

Planned models:
- Logistic Regression
- Random Forest

Metrics to evaluate:
- Precision
- Recall
- F1-score
- ROC-AUC

Final model selection will be based on recall (fraud detection priority).

## Final Recommendation

- Use the finalized non-redundant feature set as the shared starting point for both baseline models.
- Train Logistic Regression and Random Forest in the next phase and compare them under the same fraud-focused metrics.
- Prioritize fraud recall and decision-threshold calibration.

## Saved Tables

- `reports/tables/12_final_summary/project_summary.csv`
- `reports/tables/12_final_summary/final_summary_report.md`


Final summary report saved to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/12_final_summary/final_summary_report.md
